In [8]:
import pandas as pd
import asyncio
import aiohttp
import urllib.parse
import os

In [ ]:
GEOCODE_DELAY = 1.1
INPUT_FILE  = "argenprop_1776007342.tsv.tsv"
OUTPUT_FILE = "Argenprop_Lat_Lon.tsv"

CABA_LAT_MIN, CABA_LAT_MAX = -34.705, -34.527
CABA_LON_MIN, CABA_LON_MAX = -58.531, -58.335

def dentro_de_caba(lat, lon):
    return (CABA_LAT_MIN <= lat <= CABA_LAT_MAX) and (CABA_LON_MIN <= lon <= CABA_LON_MAX)

async def geocode(session, calle, altura, sem):
    if calle == "N/A" or altura == "N/A":
        return None, None
    query = f"{calle} {altura}, Ciudad Autónoma de Buenos Aires, Argentina"
    url = "https://nominatim.openstreetmap.org/search?" + urllib.parse.urlencode({
        "q": query,
        "format": "json",
        "limit": 5,
        "countrycodes": "ar",
        "viewbox": f"{CABA_LON_MIN},{CABA_LAT_MAX},{CABA_LON_MAX},{CABA_LAT_MIN}",
        "bounded": 1,
    })
    async with sem:
        await asyncio.sleep(GEOCODE_DELAY)
        try:
            async with session.get(url, headers={"User-Agent": "argenprop-scraper/1.0"}, timeout=aiohttp.ClientTimeout(total=10)) as resp:
                data = await resp.json()
                for result in data:
                    lat, lon = float(result["lat"]), float(result["lon"])
                    if dentro_de_caba(lat, lon):
                        return lat, lon
        except Exception:
            pass
    return None, None


async def run(df, desde=0):
    df = df.reset_index(drop=True)

    if pd.io.common.file_exists(OUTPUT_FILE):
        df_prev = pd.read_csv(OUTPUT_FILE, sep="\t", encoding="utf-8-sig")
        df_prev = df_prev.reset_index(drop=True)
        df["Latitud"]   = df_prev["Latitud"]   if "Latitud"   in df_prev.columns else None
        df["Longitud"]  = df_prev["Longitud"]  if "Longitud"  in df_prev.columns else None
        df["Procesada"] = df_prev["Procesada"] if "Procesada" in df_prev.columns else False
        pendientes = df[(df["Procesada"] != True) & (df.index >= desde)].index.tolist()
        print(f"📂 Retomando desde fila {desde} — {len(pendientes)} pendientes")
    else:
        df["Latitud"]   = None
        df["Longitud"]  = None
        df["Procesada"] = False
        pendientes = df[df.index >= desde].index.tolist()
        print(f"🆕 Arrancando desde fila {desde} — {len(pendientes)} propiedades")

    sem = asyncio.Semaphore(1)
    contador = 0
    total = len(pendientes)

    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=False)) as session:
        async def geo(i, row):
            nonlocal contador
            lat, lon = await geocode(session, str(row.get("Calle", "N/A")), str(row.get("Altura", "N/A")), sem)
            df.at[i, "Latitud"]   = lat
            df.at[i, "Longitud"]  = lon
            df.at[i, "Procesada"] = True
            contador += 1
            print(f"⏳ {contador}/{total} — fila {i}", end="\r")

        await asyncio.gather(*[geo(i, df.loc[i]) for i in pendientes])

    df.to_csv(OUTPUT_FILE, sep="\t", index=False, encoding="utf-8-sig")
    encontrados = df["Latitud"].notna().sum()
    print(f"\n✅ Listo! {encontrados}/{len(df)} geocodificadas dentro de CABA")
    return df

df = pd.read_csv("output/argenprop_1776007342.tsv", sep="\t", encoding="utf-8-sig")
df = await run(df)

📂 Retomando desde fila 0 — 12518 pendientes


CancelledError: 